# DMP Bridge — Batch PDF Pipeline Test

This notebook runs all PDFs in `data/raw_pdfs/` through:

```text
PDF
↓
pdfplumber extraction
↓
rule-based structure detection
↓
DMPTool narrative JSON builder
↓
debug CSV + final JSON outputs
```


## Part 1 — Imports


In [1]:
from pathlib import Path
import pandas as pd
import json

from dmpbridge.pdf.pdfplumber_extractor import save_pdfplumber_outputs
from dmpbridge.processing.structure_detector import detect_structure
from dmpbridge.processing.structure_json_builder import save_narrative_json



## Part 2 — Project paths


In [2]:
from pathlib import Path

cwd = Path.cwd()

if (cwd / "data").exists() and (cwd / "src").exists():
    project_root = cwd
else:
    project_root = cwd.parent

# Input paths
raw_pdf_dir = project_root / "data" / "reference_pdfs"
skeleton_path = project_root / "schemas" / "rda_dmp_dmptool_extension_skeleton.json"

# Output paths
pdfplumber_output_dir = project_root / "data" / "pdfplumber_extracted_blocks"
extracted_text_dir = project_root / "data" / "pdfplumber_extracted_text"
markdown_dir = project_root / "data" / "pdfplumber_extracted_markdown"

debug_output_dir = project_root / "outputs" / "debug"

for path in [
    pdfplumber_output_dir,
    extracted_text_dir,
    markdown_dir,
    debug_output_dir,
]:
    path.mkdir(parents=True, exist_ok=True)

print("Project root:", project_root)
print("Reference PDF directory exists:", raw_pdf_dir.exists())
print("Skeleton exists:", skeleton_path.exists())
print("PDFPlumber blocks directory:", pdfplumber_output_dir)
print("Extracted text directory:", extracted_text_dir)
print("Markdown directory:", markdown_dir)

Project root: c:\Users\Nahid\dmpbridge
Reference PDF directory exists: True
Skeleton exists: True
PDFPlumber blocks directory: c:\Users\Nahid\dmpbridge\data\pdfplumber_extracted_blocks
Extracted text directory: c:\Users\Nahid\dmpbridge\data\pdfplumber_extracted_text
Markdown directory: c:\Users\Nahid\dmpbridge\data\pdfplumber_extracted_markdown


## Part 3 — Find all PDFs


In [3]:
pdf_paths = sorted(raw_pdf_dir.glob("*.pdf"))

print("Number of PDFs found:", len(pdf_paths))

for pdf_path in pdf_paths:
    print("-", pdf_path.name)


Number of PDFs found: 10
- sample1.pdf
- sample10.pdf
- sample2.pdf
- sample3.pdf
- sample4.pdf
- sample5.pdf
- sample6.pdf
- sample7.pdf
- sample8.pdf
- sample9.pdf


In [4]:
# Change this index if you want to test a different PDF first.
sample_index = 0

pdf_path = pdf_paths[sample_index]

print("Testing:", pdf_path.name)

blocks = save_pdfplumber_outputs(pdf_path)

df = pd.DataFrame(blocks)

debug_columns = [
    "page",
    "line_order",
    "text",
    "avg_font_size",
    "is_bold",
    "extractor"
]

display(df[debug_columns].head(80))
print("Number of extracted lines:", len(df))

Testing: sample1.pdf
[2026-05-20 17:49:51] Extracting line-level text with pdfplumber: sample1.pdf
[2026-05-20 17:49:51] Saved line-level JSON: C:\Users\Nahid\dmpbridge\data\pdfplumber_extracted_blocks\sample1.json
[2026-05-20 17:49:51] Saved extracted text: C:\Users\Nahid\dmpbridge\data\pdfplumber_extracted_text\sample1.txt
[2026-05-20 17:49:51] Saved Markdown text: C:\Users\Nahid\dmpbridge\data\pdfplumber_extracted_markdown\sample1.md


,page,line_order,text,avg_font_size,is_bold,extractor
0,1,1,DATA MANAGEMENT AND SHARING PLAN,11.04,True,pdfplumber
1,1,2,Element 1: Data Type:,11.04,True,pdfplumber
2,1,3,A. Types and amount of scientific data expecte...,11.04,True,pdfplumber
3,1,4,This secondary data analysis project will anal...,11.04,False,pdfplumber
4,1,5,and the publicly available NHANES cohorts (wri...,11.04,False,pdfplumber
...,...,...,...,...,...,...
74,2,36,"PI, Trial & IRB Coordinator, and Statistician/...",11.04,False,pdfplumber
75,2,37,"ensure that the datasets, protocols, and codes...",11.04,False,pdfplumber
76,2,38,the main outcome manuscript has been published...,11.04,False,pdfplumber
77,2,39,University of California San Diego Library rep...,11.04,False,pdfplumber


Number of extracted lines: 79


## Part 5 — Save one sample debug CSV and narrative JSON


In [5]:
pdf_stem = pdf_path.stem

csv_output_path = debug_output_dir / f"{pdf_stem}_pdfplumber_lines.csv"

# Re-create df if this cell is run independently after blocks exists.
df = pd.DataFrame(blocks)

debug_columns = [
    "page",
    "line_order",
    "text",
    "avg_font_size",
    "is_bold",
    "extractor"
]

df[debug_columns].to_csv(
    csv_output_path,
    index=False,
    encoding="utf-8"
)

print("Saved CSV:", csv_output_path)
print("Number of extracted lines:", len(df))
print("Number of pages:", df["page"].nunique())
print("Average font size:", round(df["avg_font_size"].mean(), 2))
print("Bold lines:", df["is_bold"].sum())

Saved CSV: c:\Users\Nahid\dmpbridge\outputs\debug\sample1_pdfplumber_lines.csv
Number of extracted lines: 79
Number of pages: 2
Average font size: 11.04
Bold lines: 16


## Part 6 — Run all PDFs


In [6]:
results = []

for pdf_path in pdf_paths:
    print("\n" + "=" * 80)
    print("Processing:", pdf_path.name)

    try:
        blocks = save_pdfplumber_outputs(pdf_path)
        df = pd.DataFrame(blocks)

        pdf_stem = pdf_path.stem
        csv_output_path = debug_output_dir / f"{pdf_stem}_pdfplumber_lines.csv"

        debug_columns = [
            "page",
            "line_order",
            "text",
            "avg_font_size",
            "is_bold",
            "extractor"
        ]

        if not df.empty:
            df[debug_columns].to_csv(
                csv_output_path,
                index=False,
                encoding="utf-8"
            )

            page_count = df["page"].nunique()
            bold_line_count = df["is_bold"].sum()
            avg_font_size = round(df["avg_font_size"].mean(), 2)

        else:
            pd.DataFrame(columns=debug_columns).to_csv(
                csv_output_path,
                index=False,
                encoding="utf-8"
            )

            page_count = 0
            bold_line_count = 0
            avg_font_size = None

        result = {
            "pdf": pdf_path.name,
            "status": "success",
            "extracted_lines": len(blocks),
            "pages": page_count,
            "bold_lines": bold_line_count,
            "avg_font_size": avg_font_size,
            "csv_output": str(csv_output_path),
            "error": None
        }

        print("Extracted lines:", len(blocks))
        print("Pages:", page_count)
        print("Saved CSV:", csv_output_path)

    except Exception as e:
        result = {
            "pdf": pdf_path.name,
            "status": "failed",
            "extracted_lines": None,
            "pages": None,
            "bold_lines": None,
            "avg_font_size": None,
            "csv_output": None,
            "error": str(e)
        }

        print("FAILED:", e)

    results.append(result)

summary_df = pd.DataFrame(results)
summary_df


Processing: sample1.pdf
[2026-05-20 17:49:51] Extracting line-level text with pdfplumber: sample1.pdf
[2026-05-20 17:49:51] Saved line-level JSON: C:\Users\Nahid\dmpbridge\data\pdfplumber_extracted_blocks\sample1.json
[2026-05-20 17:49:51] Saved extracted text: C:\Users\Nahid\dmpbridge\data\pdfplumber_extracted_text\sample1.txt
[2026-05-20 17:49:51] Saved Markdown text: C:\Users\Nahid\dmpbridge\data\pdfplumber_extracted_markdown\sample1.md
Extracted lines: 79
Pages: 2
Saved CSV: c:\Users\Nahid\dmpbridge\outputs\debug\sample1_pdfplumber_lines.csv

Processing: sample10.pdf
[2026-05-20 17:49:51] Extracting line-level text with pdfplumber: sample10.pdf
[2026-05-20 17:49:51] Saved line-level JSON: C:\Users\Nahid\dmpbridge\data\pdfplumber_extracted_blocks\sample10.json
[2026-05-20 17:49:51] Saved extracted text: C:\Users\Nahid\dmpbridge\data\pdfplumber_extracted_text\sample10.txt
[2026-05-20 17:49:51] Saved Markdown text: C:\Users\Nahid\dmpbridge\data\pdfplumber_extracted_markdown\sample10.

,pdf,status,extracted_lines,pages,bold_lines,avg_font_size,csv_output,error
0,sample1.pdf,success,79,2,16,11.04,c:\Users\Nahid\dmpbridge\outputs\debug\sample1...,None
1,sample10.pdf,success,68,2,7,10.98,c:\Users\Nahid\dmpbridge\outputs\debug\sample1...,None
2,sample2.pdf,success,171,5,42,11.25,c:\Users\Nahid\dmpbridge\outputs\debug\sample2...,None
3,sample3.pdf,success,69,3,0,11.90,c:\Users\Nahid\dmpbridge\outputs\debug\sample3...,None
4,sample4.pdf,success,185,4,0,10.86,c:\Users\Nahid\dmpbridge\outputs\debug\sample4...,None
5,sample5.pdf,success,81,2,16,11.04,c:\Users\Nahid\dmpbridge\outputs\debug\sample5...,None
6,sample6.pdf,success,24,1,1,12.00,c:\Users\Nahid\dmpbridge\outputs\debug\sample6...,None
7,sample7.pdf,success,17,1,1,11.04,c:\Users\Nahid\dmpbridge\outputs\debug\sample7...,None
8,sample8.pdf,success,59,2,8,9.84,c:\Users\Nahid\dmpbridge\outputs\debug\sample8...,None
9,sample9.pdf,success,85,3,0,11.55,c:\Users\Nahid\dmpbridge\outputs\debug\sample9...,None


## Part 7 — Save batch summary


In [7]:
summary_path = debug_output_dir / "pdfplumber_batch_summary.csv"

summary_df.to_csv(
    summary_path,
    index=False,
    encoding="utf-8"
)

print("Saved batch summary:", summary_path)
display(summary_df)


Saved batch summary: c:\Users\Nahid\dmpbridge\outputs\debug\pdfplumber_batch_summary.csv


,pdf,status,extracted_lines,pages,bold_lines,avg_font_size,csv_output,error
0,sample1.pdf,success,79,2,16,11.04,c:\Users\Nahid\dmpbridge\outputs\debug\sample1...,None
1,sample10.pdf,success,68,2,7,10.98,c:\Users\Nahid\dmpbridge\outputs\debug\sample1...,None
2,sample2.pdf,success,171,5,42,11.25,c:\Users\Nahid\dmpbridge\outputs\debug\sample2...,None
3,sample3.pdf,success,69,3,0,11.90,c:\Users\Nahid\dmpbridge\outputs\debug\sample3...,None
4,sample4.pdf,success,185,4,0,10.86,c:\Users\Nahid\dmpbridge\outputs\debug\sample4...,None
5,sample5.pdf,success,81,2,16,11.04,c:\Users\Nahid\dmpbridge\outputs\debug\sample5...,None
6,sample6.pdf,success,24,1,1,12.00,c:\Users\Nahid\dmpbridge\outputs\debug\sample6...,None
7,sample7.pdf,success,17,1,1,11.04,c:\Users\Nahid\dmpbridge\outputs\debug\sample7...,None
8,sample8.pdf,success,59,2,8,9.84,c:\Users\Nahid\dmpbridge\outputs\debug\sample8...,None
9,sample9.pdf,success,85,3,0,11.55,c:\Users\Nahid\dmpbridge\outputs\debug\sample9...,None
